# YOLOv10 Training — Learning Rate 0.0001

**MCE 415 — Computer Vision | Group 8 | FUT Minna**

This notebook trains YOLOv10-M on the unified road anomaly dataset with **LR = 0.0001**.  
Run all cells top-to-bottom on a **Kaggle T4 GPU** notebook.

| Setting | Value |
|---|---|
| Model | YOLOv10-M (COCO pretrained) |
| Learning Rate | **0.0001** |
| Epochs | 100 |
| Batch Size | 16 |
| Optimizer | AdamW |

---
## 1. Environment Setup

In [ ]:
# Install dependencies — ultralytics for YOLOv10, gdown for Google Drive download
# Quotes prevent the shell from interpreting >= as a redirect operator
!pip install -q "ultralytics>=8.2.50" pyyaml seaborn gdown

# Verify GPU is available
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device:      {torch.cuda.get_device_name(0)}")
    print(f"GPU memory:      {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected!")
    print("Enable GPU in Kaggle: Settings > Accelerator > GPU T4 x2")

In [ ]:
# Core imports
import os
import yaml
import shutil
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from datetime import datetime

import torch
from ultralytics import YOLO

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Ultralytics version: {__import__('ultralytics').__version__}")
print(f"Setup complete — {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

---
## 2. Download & Prepare Dataset

In [ ]:
import gdown
import zipfile
import glob as _glob

# ============================================================
# GOOGLE DRIVE DOWNLOAD CONFIGURATION
# ============================================================
GDRIVE_FILE_ID = "18JDX57ppDZpPvWUaeFFkXgHM28NgIzV5"
ZIP_PATH = Path("/kaggle/working/unified_dataset.zip")
DATASET_ROOT = Path("/kaggle/working/unified_dataset")
EXTRACT_TMP = Path("/kaggle/working/_extract_tmp")

# ----------------------------------------------------------
# Step 1: Download zip from Google Drive (skip if cached)
# ----------------------------------------------------------
if not ZIP_PATH.exists():
    print("Downloading dataset from Google Drive...")
    gdown.download(id=GDRIVE_FILE_ID, output=str(ZIP_PATH), quiet=False)
    if ZIP_PATH.stat().st_size < 1_000_000:
        ZIP_PATH.unlink()
        raise RuntimeError(
            "Downloaded file is too small — likely a Google Drive error page.\n"
            "Make sure the file is shared with 'Anyone with the link'."
        )
    print(f"Downloaded: {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1e6:.1f} MB)")
else:
    print(f"Zip already exists — skipping download ({ZIP_PATH.stat().st_size / 1e6:.1f} MB)")

# ----------------------------------------------------------
# Step 2: Extract into temp dir, auto-detect layout, move to
# DATASET_ROOT.  Handles BOTH possible structures:
#   Layout A (standard YOLO): train/images, val/images, ...
#   Layout B (inverted):      images/train, images/val, ...
# ----------------------------------------------------------
already_extracted = (
    (DATASET_ROOT / "train" / "images").exists() or
    (DATASET_ROOT / "images" / "train").exists()
)

if not already_extracted:
    if DATASET_ROOT.exists():
        shutil.rmtree(DATASET_ROOT)
    if EXTRACT_TMP.exists():
        shutil.rmtree(EXTRACT_TMP)
    EXTRACT_TMP.mkdir(parents=True)

    print("Extracting dataset into temp directory...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(str(EXTRACT_TMP))

    candidates_a = _glob.glob(str(EXTRACT_TMP / "**" / "train" / "images"), recursive=True)
    candidates_b = _glob.glob(str(EXTRACT_TMP / "**" / "images" / "train"), recursive=True)

    if candidates_a:
        actual_root = Path(candidates_a[0]).parent.parent
        print(f"Detected Layout A (train/images): {actual_root}")
    elif candidates_b:
        actual_root = Path(candidates_b[0]).parent.parent
        print(f"Detected Layout B (images/train): {actual_root}")
    else:
        with zipfile.ZipFile(ZIP_PATH, "r") as zf:
            all_names = zf.namelist()
        print(f"ERROR: Could not detect dataset layout! Zip has {len(all_names)} entries.")
        print("First 50 entries:")
        for n in all_names[:50]:
            print(f"  {n}")
        raise FileNotFoundError("Could not locate train/images or images/train in zip")

    shutil.move(str(actual_root), str(DATASET_ROOT))
    print(f"Moved → {DATASET_ROOT}")

    if EXTRACT_TMP.exists():
        shutil.rmtree(EXTRACT_TMP)
else:
    print("Dataset already extracted — skipping")

# ----------------------------------------------------------
# Step 3: Detect layout and set path helpers
# ----------------------------------------------------------
if (DATASET_ROOT / "train" / "images").exists():
    LAYOUT = "A"
    def img_dir(split): return DATASET_ROOT / split / "images"
    def lbl_dir(split): return DATASET_ROOT / split / "labels"
    yaml_train, yaml_val, yaml_test = "train/images", "val/images", "test/images"
elif (DATASET_ROOT / "images" / "train").exists():
    LAYOUT = "B"
    def img_dir(split): return DATASET_ROOT / "images" / split
    def lbl_dir(split): return DATASET_ROOT / "labels" / split
    yaml_train, yaml_val, yaml_test = "images/train", "images/val", "images/test"
else:
    raise RuntimeError(f"Cannot detect layout in {DATASET_ROOT}. Contents: {list(DATASET_ROOT.iterdir())}")

print(f"Layout: {LAYOUT} ({'train/images' if LAYOUT == 'A' else 'images/train'})")

# ----------------------------------------------------------
# Step 4: Ensure data.yaml exists with correct paths
# ----------------------------------------------------------
DATA_YAML_PATH = DATASET_ROOT / "data.yaml"
if not DATA_YAML_PATH.exists():
    print("data.yaml not found — creating it now...")
    data_yaml_content = {
        "path": str(DATASET_ROOT),
        "train": yaml_train,
        "val": yaml_val,
        "test": yaml_test,
        "nc": 3,
        "names": ["Pothole", "Speedbump", "Crack"],
    }
    with open(DATA_YAML_PATH, "w") as f:
        yaml.dump(data_yaml_content, f, default_flow_style=False, sort_keys=False)
    print(f"Created: {DATA_YAML_PATH}")
else:
    with open(DATA_YAML_PATH, "r") as f:
        data_cfg = yaml.safe_load(f)
    data_cfg["path"] = str(DATASET_ROOT)
    data_cfg["train"] = yaml_train
    data_cfg["val"] = yaml_val
    data_cfg["test"] = yaml_test
    with open(DATA_YAML_PATH, "w") as f:
        yaml.dump(data_cfg, f, default_flow_style=False, sort_keys=False)
    print("data.yaml found — updated paths")

# ----------------------------------------------------------
# Final verification
# ----------------------------------------------------------
assert img_dir("train").exists(), f"Missing: {img_dir('train')}"
assert lbl_dir("train").exists(), f"Missing: {lbl_dir('train')}"
assert img_dir("val").exists(), f"Missing: {img_dir('val')}"
assert lbl_dir("val").exists(), f"Missing: {lbl_dir('val')}"

if not img_dir("test").exists():
    print(f"Note: test split not found (OK if you only use train/val)")

assert DATA_YAML_PATH.exists()
print(f"\nContents: {sorted(p.name for p in DATASET_ROOT.iterdir())}")
print("All required directories verified ✓")

In [ ]:
# Point WORK_DIR to the extracted dataset and update data.yaml
WORK_DIR = DATASET_ROOT

DATA_YAML = WORK_DIR / "data.yaml"
with open(DATA_YAML, "r") as f:
    data_cfg = yaml.safe_load(f)

data_cfg["path"] = str(WORK_DIR)
with open(DATA_YAML, "w") as f:
    yaml.dump(data_cfg, f, default_flow_style=False, sort_keys=False)

CLASS_NAMES = data_cfg["names"]
print(f"Working directory: {WORK_DIR}")
print(f"Classes: {CLASS_NAMES}")
print(f"data.yaml path updated to: {data_cfg['path']}")

In [ ]:
# Sanity check — count images and annotations per split
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}

print(f"Classes: {CLASS_NAMES}")
print(f"Number of classes: {data_cfg['nc']}\n")

for split in ["train", "val", "test"]:
    _img = img_dir(split)
    _lbl = lbl_dir(split)

    if not _img.exists():
        print(f"{split.upper():>5}: (not present — skipping)")
        print()
        continue

    n_images = len([f for f in _img.iterdir() if f.suffix.lower() in IMG_EXTS])

    class_counts = defaultdict(int)
    if _lbl.exists():
        for lbl_file in _lbl.glob("*.txt"):
            with open(lbl_file) as f:
                for line in f:
                    if line.strip():
                        cls_id = int(line.strip().split()[0])
                        class_counts[cls_id] += 1

    total_ann = sum(class_counts.values())
    print(f"{split.upper():>5}: {n_images:>5} images, {total_ann:>5} annotations")
    for cls_id in sorted(class_counts):
        print(f"        Class {cls_id} ({CLASS_NAMES[cls_id]}): {class_counts[cls_id]}")
    print()

---
## 3. Training — LR = 0.0001

In [ ]:
# ============================================================
# TRAINING CONFIGURATION
# ============================================================

LEARNING_RATE = 0.0001
MODEL_WEIGHTS = "yolov10m.pt"
PROJECT_DIR = Path("/kaggle/working/runs")
RUN_NAME = f"lr_{LEARNING_RATE}"

TRAIN_CONFIG = {
    "imgsz": 640,
    "epochs": 100,
    "batch": 16,
    "optimizer": "AdamW",
    "seed": SEED,
    "device": 0,
    "workers": 4,
    "exist_ok": True,
    "verbose": True,
    "plots": True,
    "save": True,
    "val": True,
    # -- Enabled augmentations (project brief: Mosaic, MixUp, CutMix ONLY) --
    "mosaic": 1.0,
    "mixup": 0.15,
    "cutmix": 0.1,          # CutMix augmentation
    # -- Disabled augmentations (everything else turned off) --
    "hsv_h": 0.0, "hsv_s": 0.0, "hsv_v": 0.0,
    "degrees": 0.0, "translate": 0.0, "scale": 0.0,
    "shear": 0.0, "perspective": 0.0,
    "flipud": 0.0, "fliplr": 0.0,
    "erasing": 0.0, "crop_fraction": 1.0,
    # -- Reproducibility --
    "deterministic": True,
}

print(f"Learning Rate: {LEARNING_RATE}")
print(f"Model: {MODEL_WEIGHTS}")
print(f"Run will be saved to: {PROJECT_DIR / RUN_NAME}")

In [ ]:
# Load fresh COCO-pretrained model and train
print("=" * 70)
print(f"TRAINING: Learning Rate = {LEARNING_RATE}")
print("=" * 70)

model = YOLO(MODEL_WEIGHTS)

results = model.train(
    data=str(DATA_YAML),
    lr0=LEARNING_RATE,
    lrf=0.01,
    project=str(PROJECT_DIR),
    name=RUN_NAME,
    **TRAIN_CONFIG
)

print(f"\nTraining complete for LR={LEARNING_RATE}")
print(f"Best weights: {PROJECT_DIR / RUN_NAME / 'weights' / 'best.pt'}")
print(f"Results CSV:  {PROJECT_DIR / RUN_NAME / 'results.csv'}")

---
## 4. Package Results for Download

In [ ]:
# Package the training output into a zip for download
import zipfile

run_dir = PROJECT_DIR / RUN_NAME
zip_name = f"Group8_LR_{LEARNING_RATE}_results.zip"
zip_path = Path(f"/kaggle/working/{zip_name}")

print(f"Packaging results into {zip_name}...")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(run_dir):
        for file in files:
            file_path = Path(root) / file
            arcname = file_path.relative_to(run_dir.parent)
            zf.write(file_path, arcname)

size_mb = zip_path.stat().st_size / (1024 * 1024)
print(f"\nDone! Zip created: {zip_path} ({size_mb:.1f} MB)")
print("Download this from Kaggle's Output tab.")
print(f"\nContents include:")
print(f"  - weights/best.pt (best model checkpoint)")
print(f"  - weights/last.pt (final epoch checkpoint)")
print(f"  - results.csv (per-epoch metrics)")
print(f"  - Training plots (confusion matrix, PR curves, etc.)")

In [ ]:
# Quick peek at this run's results
csv_path = run_dir / "results.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    best_idx = df["metrics/mAP50(B)"].idxmax()
    best_row = df.loc[best_idx]
    print(f"\n{'=' * 50}")
    print(f"LR = {LEARNING_RATE} — BEST EPOCH SUMMARY")
    print(f"{'=' * 50}")
    print(f"Best Epoch:     {int(best_row['epoch']) + 1}")
    print(f"mAP@0.5:        {best_row['metrics/mAP50(B)']:.4f}")
    print(f"mAP@0.5:0.95:   {best_row['metrics/mAP50-95(B)']:.4f}")
    print(f"Precision:      {best_row['metrics/precision(B)']:.4f}")
    print(f"Recall:         {best_row['metrics/recall(B)']:.4f}")
else:
    print("WARNING: results.csv not found — training may have failed")

# Cleanup GPU memory
del model
torch.cuda.empty_cache()
print("\nGPU memory released. You can close this notebook.")